In [11]:
import zipfile
import os
import cv2
import numpy as np

zip_path = "/content/archive (21).zip"
extract_destination = "brain_tumor_dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_destination)

print("Dataset Extracted Successfully")

# Count the number of extracted files/images
extracted_image_count = 0
for dirpath, dirnames, filenames in os.walk(extract_destination):
    extracted_image_count += len(filenames)

print("Total images loaded:", extracted_image_count)

Dataset Extracted Successfully
Total images loaded: 506


In [12]:
import os
import cv2
import numpy as np

images = []
labels = []

dataset_path = "brain_tumor_dataset"

for label_name in ["yes", "no"]:

    folder_path = os.path.join(dataset_path, label_name)

    for file in os.listdir(folder_path):

        img_path = os.path.join(folder_path, file)

        img = cv2.imread(img_path)

        if img is not None:

            # Resize
            img = cv2.resize(img, (128, 128))

            # Normalize
            img = img / 255.0

            images.append(img)

            if label_name == "yes":
                labels.append(1)
            else:
                labels.append(0)

X = np.array(images)
y = np.array(labels)

print("Images Shape:", X.shape)
print("Labels Shape:", y.shape)

Images Shape: (253, 128, 128, 3)
Labels Shape: (253,)


In [13]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training Images:", X_train.shape)
print("Testing Images :", X_test.shape)

Training Images: (202, 128, 128, 3)
Testing Images : (51, 128, 128, 3)


In [14]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential()

# Convolution Layer 1
model.add(Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)))
model.add(MaxPooling2D(pool_size=(2,2)))

# Convolution Layer 2
model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))

# Convolution Layer 3
model.add(Conv2D(128, (3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))

# Flatten
model.add(Flatten())

# Dense Layers
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

# Output Layer
model.add(Dense(1, activation='sigmoid'))

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_6 (Conv2D)               │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_8 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_8 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,304,769 (12.61 MB)

 Trainable params: 3,304,769 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 8s 855ms/step - accuracy: 0.7277 - loss: 0.6081 - val_accuracy: 0.6863 - val_loss: 0.6606
Epoch 2/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 7s 979ms/step - accuracy: 0.7228 - loss: 0.5611 - val_accuracy: 0.7647 - val_loss: 0.5064
Epoch 3/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 979ms/step - accuracy: 0.8168 - loss: 0.4882 - val_accuracy: 0.7647 - val_loss: 0.4975
Epoch 4/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 6s 811ms/step - accuracy: 0.7970 - loss: 0.4511 - val_accuracy: 0.8039 - val_loss: 0.4759
Epoch 5/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 11s 857ms/step - accuracy: 0.8465 - loss: 0.3819 - val_accuracy: 0.7843 - val_loss: 0.4451
Epoch 6/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.8465 - loss: 0.3678 - val_accuracy: 0.7647 - val_loss: 0.4464
Epoch 7/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 9s 882ms/step - accuracy: 0.8762 - loss: 0.3402 - val_accuracy: 0.7647 - val_loss: 0.4630
Epoch 8/10
7/7 ━━━━━━━━━━━━━━━━━━━━ 10s 815ms/step - accuracy: 0.9059 - loss: 0.2517 - val_accuracy: 0.7647 - val_loss

In [16]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)

print("Test Accuracy:", test_accuracy)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 173ms/step - accuracy: 0.7843 - loss: 0.5823
Test Accuracy: 0.7843137383460999


In [17]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# Predict probabilities
y_pred = model.predict(X_test)

# Convert probabilities to 0 or 1
y_pred = (y_pred > 0.5).astype(int)

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 289ms/step

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 214ms/step
Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.50      0.65        20
           1       0.75      0.97      0.85        31

    accuracy                           0.78        51
   macro avg       0.83      0.73      0.75        51
weighted avg       0.81      0.78      0.77        51

Confusion Matrix:
[[10 10]
 [ 1 30]]


In [19]:
import cv2
import numpy as np

image_path = "/content/WhatsApp Image 2026-06-15 at 2.03.13 AM.jpeg"   # Change path

img = cv2.imread(image_path)
img = cv2.resize(img, (128,128))
img = img / 255.0

img = np.expand_dims(img, axis=0)

prediction = model.predict(img)

if prediction[0][0] > 0.5:
    print("Tumor Detected")
else:
    print("No Tumor Detected")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
No Tumor Detected
